# Notebook for EDA

## 1. Read in cleaned data files

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
stadium = pd.read_csv('data/processed/clean_stadium.csv')
fanbase = pd.read_csv('data/processed/fanbase_clean.csv')
merch = pd.read_csv('data/processed/cleaned_merch.csv')
fan_merch_merged = pd.read_csv('data/processed/merch_fanbase_merged.csv')

## Missing values in Merch

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(merch.isnull(), cbar=False, cmap='viridis')
plt.title("Missing Data Heatmap")
plt.show()

### Missing At Random (MAR):
I already know from looking at the data that missing values in arrival date align with merch bought in store, and missing sizes align with products that do not require sizes (such as caps, mugs, posters) but this would be good to confirm.

In [ ]:
data = {
    'Product': merch['Product_ID'].unique(),
    'Item': merch['Item_Name'].unique()
}
product_table = pd.DataFrame(data)
product_table

In [ ]:
size_ids = [10000002, 10000003, 10000004, 10000005, 10000006, 10000013, 10000014, 10000015, 10000016, 10000017]
merch['should_have_size'] = merch['Product_ID'].isin(size_ids)
merch['size_missing'] = merch['Size'].isna()

In [ ]:
mismatches = merch[merch['should_have_size'] & merch['size_missing']]
mismatches.shape

OK, we confirm that all missing values in the size column are due to products not needing sizes.

In [ ]:
merch = merch.drop(['size_missing', 'should_have_size'], axis=1)

Confirm that all missing values in arrival date are due to products being bought in store and not online:

In [ ]:
merch['arrival_missing'] = merch['Arrival_Date'].isna()
merch['should_have_arrival'] = merch['Channel'] != 'Team Store'

In [ ]:
arrival_mismatches = merch[merch['arrival_missing'] & merch['should_have_arrival']]
arrival_mismatches.shape

In [ ]:
store_date_mismatches = merch[~merch['arrival_missing'] & (merch['Channel'] == 'Team Store')]
store_date_mismatches.shape

In [ ]:
merch = merch.drop(['arrival_missing', 'should_have_arrival'], axis=1)
merch.head()

Ok, we have confirmed that all missing values are logical and values that should have input, do have input

## EDA: Stadium

1. Revenue Decomposition by Month:
- potentially reflects seasonality of games
- revenue loss months likely due to maintenace costs not being balanced out by ticket/merch/food sales
- interesting V shaped trend with June being a revenue loss month
    - We should look into this as an avenue to mitigate loss

In [ ]:
overall_revenue = alt.Chart(stadium).mark_bar().encode(
    x='Month',
    y='sum(Revenue)',
)
overall_revenue

2. Revenue breakdown by Source (monthly)
- sources of loss: Maintenance, Insurance, Staff, Utilities
- main sources of gain: Upper Bowl ticket sales, food
- potential sources to explore: Concerts (seems to line up with off season)

In [ ]:
rev_gain_or_loss = alt.Chart(stadium).mark_bar().encode(
    x='Month:O',
    y='Revenue:Q',
    color='Revenue_Flag:N',
).facet(
    'Source',
    columns = 1
)
rev_gain_or_loss

3. Revenue trends, all sources
- Food sales align with lower bowl ticket sales (makes sense)
- Concert sales peak in July and in December
- highest maintenance costs in January and somewhat in December
- most other revenue sources seem consistent over time

In [ ]:
revenue_by_source = alt.Chart(stadium).mark_line().encode(
    x='Month',
    y='Revenue',
    color='Source'
)
revenue_by_source

4. Heatmap of monthly revenue, by source

In [ ]:
alt.Chart(stadium).mark_rect().encode(
    x='Month:O',
    y='Source:N',
    color='Revenue:Q'
)

5. Distribution of revenue source data
- some outliers in Maitnenance, Staff, Advertising - attributable to seasonal peaks
- as previously seen, lower bowl and food have the largest spread of data

In [ ]:
alt.Chart(stadium).mark_boxplot().encode(
    x='Source:N',
    y='Revenue:Q'
)

In [ ]:
total = sum(stadium['Revenue'])
total

In [ ]:
grouped = stadium.groupby('Source', as_index=False).agg({'Revenue': 'sum'})
grouped.rename(columns={'Revenue': 'Source Earnings'}, inplace=True)

max(grouped['Source Earnings'])

In [ ]:
grouped

### Insights:
- primary sources of income are from Lower Bowl and Food
- Upper Bowl is significantly lower earning than LB
- The highest source of loss is Staff
- February and October have the overall highest profit
- January, June, November, December are overall losses
- insurance seems to be the only fixed cost (arguably, staff as well, but it does vary month by month)

1. The overall revenue across the 2024 period was: $13,233,516.00
2. The highest earning month was February with: $3'956'727.00
3. The lowest earning month was November with: -$2'576'211.00 (followed closely by December)
4. Total staff costs were: -$39'646'000.00
5. Total Lower Bowl earnings were: $24'669'304.00

## EDA: Fanbase

In [ ]:
alt.data_transformers.disable_max_rows()

1. Games Attended total distributions
- most fans attend between 0-10 games, very few attend 15-30
- no one attended between 10-15 games?

In [ ]:
alt.Chart(fanbase).mark_bar().encode(
    x=alt.X('Games_Attended:Q', bin=True),
    y='count()',
    tooltip=['Games_Attended']
).properties(title='Distribution of Games Attended')

2. Distribution of fanbase membership by age group

In [ ]:
alt.Chart(fanbase).mark_bar().encode(
    x='Customer_Age_Group:N',
    y='count()',
    color='Customer_Age_Group:N'
).properties(title='Customer Age Group Distribution')

3. Seasonal pass adoption, by age group
- no apparent trends, proportional to density of each age group

In [ ]:
alt.Chart(fanbase).mark_bar().encode(
    x='Customer_Age_Group:N',
    y='count()',
    color='Seasonal_Pass:N'
).properties(title='Seasonal Pass Adoption by Age Group')

4. Heatmap of age group vs region belonging
- the vast majority of fans are domestic, across all age groups

In [ ]:
alt.Chart(fanbase).mark_rect().encode(
    x='Customer_Age_Group:N',
    y='Region_Type:N',
    color='count()',
    tooltip=['Customer_Age_Group', 'Region_Type']
).properties(title='Customer Distribution by Region Type and Age Group')

5. Density of pass holders and game attendance
- no overlap at all! If they are a season pass holder they see between 15-30 games
- explains the bimodality in the earlier histogram

In [ ]:
alt.Chart(fanbase).transform_density(
    density='Games_Attended',
    groupby=['Seasonal_Pass'],
    as_=['Games_Attended', 'density']
).mark_area(opacity=0.5).encode(
    x=alt.X('Games_Attended:Q', title='Games Attended'),
    y=alt.Y('density:Q', title='Density'),
    color=alt.Color('Seasonal_Pass:N', title='Pass Status')
).properties(
    title='Distribution of Games Attended by Pass Status',
)

6. Region Type composition

In [ ]:
alt.Chart(fanbase).mark_arc().encode(
    theta='count()',
    color='Region_Type:N',
    tooltip=['Region_Type']
).properties(
    title='Region Type Breakdown',
    width=200,
    height=200
)

7. Specific customer region distribution

In [ ]:
alt.Chart(fanbase).mark_bar().encode(
    x='Customer_Region:N',
    y='count()',
    color='Customer_Region:N'
).properties(title='Customer Region Distribution')

In [ ]:
fanbase['Seasonal_Pass'].value_counts()

### Insights:
- The vast majority of memberships come from Domestic fans
- The largest demographic of fans are 18-25 year olds
- Most people attend 0-10 games and are non season pass holders
- People who see 15-20 games are more likely to be season pass holders
- Seasonal pass holders make up about 7% of the fanbase
- Of the international fans - the majority come from the US, followed by India

## EDA: Merch

1. Age distribution of merch purchases, plus usage of channels by age group
- as expected, largest group is 18-25 year olds
- online vs team store usage is proportional to demographic quantity

In [ ]:
alt.Chart(merch).mark_bar().encode(
    x='Customer_Age_Group',
    y='count()',
    color='Channel:N'
).properties(title='Age distribution by sales channel')

2. Channel Performance
- Vast majority purchase online

In [ ]:
alt.Chart(merch).mark_bar().encode(
    x='Channel:N',
    y='count()',
    color='Channel:N'
).properties(title='Channel of Purchase')

3. Purchases through promotion and without
- most are without, but a fair amount of purchases happen through a promotion

In [ ]:
alt.Chart(merch).mark_bar().encode(
    x='Promotion:N',
    y='count()',
    color='Promotion:N'
).properties(title='Distribution of Promotion Based Purchases')

4. Sales by item, including breakdown by channel and whether or not purchased with a promotion
- Jerseys sell the most, followed by hoodies
- promotions are only used with online purchases

In [ ]:
alt.Chart(merch).mark_bar().encode(
    x=alt.X('count()', title='Number of Sales'),
    y=alt.Y('Item_Category:N', sort='-x'),
    color='Channel',
    column=alt.Column('Promotion:N', title='Promotion')
).properties(
    title='Sales count by Item Category, Channel, and Promotion'
)

5. Customer Region

In [ ]:
alt.Chart(merch).mark_arc().encode(
    theta='count()',
    color='Customer_Region:N'
).properties(title='Customer region share')

6. Purchases through a promotion by age group
- proportional, no obvious trends

In [ ]:
alt.Chart(merch).mark_bar().encode(
    x='Customer_Age_Group:N',
    y='count()',
    color='Promotion:N',
).properties(title='Promotional vs non-promotional sales by age group and channel')

In [ ]:
merch['Selling_Date'] = pd.to_datetime(merch['Selling_Date'])
merch['Month'] = merch['Selling_Date'].dt.to_period('M').astype(str)

7. Monthly sales trends
- peak sales occur in March - could be due to beginning of season
- dip in May and in September

In [ ]:
alt.Chart(merch).mark_line(point=True).encode(
    x='Month:T',
    y='count()',
    color='Channel:N'
).properties(title='Monthly sales trends by channel')

In [ ]:
merch['Arrival_Date'] = pd.to_datetime(merch['Arrival_Date'])
merch['Selling_Date'] = pd.to_datetime(merch['Selling_Date'])
merch['delivery_days'] = (merch['Arrival_Date'] - merch['Selling_Date']).dt.days

8. How long it usually takes to deliver items (in days)
- most items take about 8 days to be delivered
- some delays occurring with 9-10 days shipping time
- no obvious discrepancies between domestic and international arrival dates

In [ ]:
alt.Chart(merch.dropna(subset=['delivery_days'])).mark_bar().encode(
    x=alt.X('delivery_days:Q', bin=alt.Bin(maxbins=20), title='Days to arrival'),
    y='count()',
    color='Customer_Region'
).properties(title='Distribution of delivery times')

9. Heatmap of product preference by age group
- Jerseys are the most popular item for all age groups
- no age group specific trends

In [ ]:
heatmap_data = merch.groupby(['Customer_Age_Group', 'Item_Category']).size().unstack(fill_value=0)
heatmap_data_reset = heatmap_data.reset_index().melt(id_vars='Customer_Age_Group', var_name='Item_Category', value_name='Volume')

alt.Chart(heatmap_data_reset).mark_rect().encode(
    x=alt.X('Item_Category:N', title='Product Category'),
    y=alt.Y('Customer_Age_Group:N', title='Age Group'),
    color=alt.Color('Volume:Q', scale=alt.Scale(scheme='blues'), title='Preference Volume'),
    tooltip=['Customer_Age_Group', 'Item_Category', 'Volume']
).properties(
    title='Age Group Preferences by Product Category',
    width=500,
    height=300
)

### Insights:
- 18-25 year olds are the largest demographic for merch purchases, followed by 26-40 year olds*
- the majority of purchases happen online
- majority of purchases do not happen with a promotion, but a reasonably large amount are through a promotion
- all purchases through a promotion happen online (makes sense if promotions are online ads)
- the vast majority of merch purchases occur domestically*
- there does not seem to be a trend between age demographic and use of promotions
- major peak in purchases around March, with a dip in September
- most purchases take around 8 days to be delivered, with some delays leading to 9-10 day delivery

*note these values have been transformed based on the assumptions made in the data cleaning - exact values may not be accurate

## EDA merged merch/fanbase data


1. Venn Diagram of fans who attended a game and purchased merch
- promising for generating probability statements

In [ ]:
from matplotlib_venn import venn2
import matplotlib.pyplot as plt

# Define sets based on Member_ID
attended = set(fan_merch_merged.loc[fan_merch_merged['Games_Attended'].notna() & (fan_merch_merged['Games_Attended'] > 0), 'Member_ID'])
purchased = set(fan_merch_merged.loc[fan_merch_merged['Item_Category'].notna(), 'Member_ID'])

# Plot Venn diagram
plt.figure(figsize=(6, 4))
venn2([attended, purchased], set_labels=('Attended Event', 'Purchased Merch'))
plt.title('Overlap Between Attendance and Merchandise Purchase')
plt.show()

2. Calculating % of members who attending 0 games: 0! All member attended at least 1 game

In [ ]:

total_members = fan_merch_merged['Member_ID'].nunique()

inactive_members = fan_merch_merged[(fan_merch_merged['Games_Attended'].isna()) | (fan_merch_merged['Games_Attended'] == 0)]['Member_ID'].nunique()


inactive_pct = (inactive_members / total_members) * 100
print(f"{inactive_pct:.2f}% of members attended 0 games.")

3. Trying to find parallel coordinates between age group, region, and games attended
- there's actually no trends, like what the hell is this dataset

In [ ]:

parallel_df = fan_merch_merged[['Customer_Age_Group', 'Customer_Region', 'Games_Attended', 'Purchased_Merch']].dropna()


parallel_df['Age_Code'] = parallel_df['Customer_Age_Group'].astype('category').cat.codes
parallel_df['Region_Code'] = parallel_df['Customer_Region'].astype('category').cat.codes


fig = px.parallel_coordinates(
    parallel_df,
    dimensions=['Age_Code', 'Region_Code', 'Games_Attended', 'Purchased_Merch'],
    color='Games_Attended',
    labels={
        'Age_Code': 'Age Group',
        'Region_Code': 'Region',
        'Purchased_Merch': 'Purchased Merch',
        'Games_Attended': 'Games Attended'
    },
    color_continuous_scale='Viridis'
)

fig.show()

In [ ]:
parallel_df.head()

In [ ]:
revenue_by_member = fan_merch_merged.groupby('Member_ID')['Unit_Price'].sum().reset_index(name='Revenue')

revenue_by_member 